# Person A — day six: connect the scoring and cache paths

Use a **fresh Colab T4 runtime**. This run deliberately changes to B's tested Torch/HQQ versions and FP32 weights. Earlier FP16 results remain a separate configuration.

Goal: test token boundaries, explicit valid-token masks, empty-quantized-cache prefill, real reconstructed K/V reaching Llama's SDPA interface, instrumentation neutrality, fresh-cache repetition, and generation replay.

We use the pinned full checkpoint and two artificial prompts, not unseen TOFU authors. There are no labels to fill. This is a partial integration diagnostic, not a recovery experiment or a final validation sign-off.

Files needed: this notebook and `day6-integration-input-bundle.zip`.

## 1. Check the accelerator
Choose **Runtime → Change runtime type → T4 GPU**. Then run this cell.

In [ ]:
import subprocess, sys
print(sys.version)
subprocess.run(["nvidia-smi"], check=True)


## 2. Install in this fresh runtime, then restart
This setup is required for a fresh default Colab runtime. It removes preinstalled vision/audio packages that caused B's Torch/Transformers import conflict; neither is used here. Torch CUDA 12.8, Transformers and HQQ are pinned explicitly.

After success, use **Runtime → Restart session**. Then continue at section 3; do not rerun installation unless it failed. Preserve any install error and do not switch versions to make a check pass.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y",
                "torchvision", "torchaudio"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install",
                "transformers==5.5.4", "hqq==0.2.8.post1",
                "accelerate==1.13.0", "huggingface-hub==1.33.0",
                "numpy==2.1.3"], check=True)
print("Installation completed. Restart the session now, then continue at section 3.")


## 3. Verify the restarted environment

In [ ]:
import torch, transformers, importlib.metadata
assert torch.cuda.is_available(), "Select a T4 GPU."
assert torch.__version__.split("+")[0] == "2.9.1", "Restart after package setup."
assert torch.version.cuda == "12.8", "Use the CUDA 12.8 Torch wheel from section 2."
assert transformers.__version__ == "5.5.4"
assert importlib.metadata.version("hqq") == "0.2.8.post1"
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__, "Transformers:", transformers.__version__,
      "HQQ:", importlib.metadata.version("hqq"))


## 4. Upload the preparation ZIP
Upload `day6-integration-input-bundle.zip`, not an earlier output ZIP. The hash check prevents a mismatched runner. A new timestamped run directory protects earlier outputs.

In [ ]:
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import zipfile, io, hashlib, json
uploaded = files.upload()
NAME = "day6-integration-input-bundle.zip"
assert NAME in uploaded, "Upload the day-six preparation ZIP."
payload = uploaded[NAME]
assert hashlib.sha256(payload).hexdigest() == "01387cf96c073dccbabded842dd3c3bb4e3f4398ba924c2cb5cf254ab5d08454", "Bundle differs from this notebook."
RUN_ID = "day6_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
BASE = Path("/content/kv-day6")
SOURCE = BASE / (RUN_ID + "_input")
SOURCE.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(io.BytesIO(payload)) as z:
    assert set(z.namelist()) == {"day6_integration.py", "preparation-manifest.json"}
    for name in z.namelist():
        (SOURCE / name).write_bytes(z.read(name))
PREP = json.loads((SOURCE / "preparation-manifest.json").read_text())
assert hashlib.sha256((SOURCE / "day6_integration.py").read_bytes()).hexdigest() == PREP["runner_sha256"]
OUT = BASE / RUN_ID
print("Run:", RUN_ID)


## 5. Run the checks
The pinned 1B full model loads once in FP32. Progress prints after each prompt/bit-width pair. Downloads and first model loading may take longer than inference.

`checks_passed_review_pending` means the programmed checks passed; both authors must still inspect the report. `checks_failed_review_required` or a runtime exception is useful diagnostic evidence. Do not relax tolerances or change the model. If this cell errors, run section 6 to save partial artifacts.

In [ ]:
import importlib.util, traceback
spec = importlib.util.spec_from_file_location("kv_day6_runner", SOURCE / "day6_integration.py")
runner = importlib.util.module_from_spec(spec)
try:
    spec.loader.exec_module(runner)
    result = runner.main(OUT)
    print("STATUS:", result["status"])
    print("FAILED CHECKS:", result.get("failed_checks", []))
except Exception:
    OUT.mkdir(parents=True, exist_ok=True)
    (OUT / "notebook-error.txt").write_text(traceback.format_exc())
    raise
finally:
    if OUT.exists():
        (OUT / "preparation-manifest.json").write_text(json.dumps(PREP, indent=2))
        (OUT / "input-bundle-sha256.txt").write_text(hashlib.sha256(payload).hexdigest()+"\n")


## 6. Download the output, including partial results on failure
Return this ZIP for review. No manual correctness annotation is needed. Keep the archive before ending the temporary runtime.

In [ ]:
import shutil
assert OUT.exists(), "Run section 5 first."
archive = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT))
print("Output:", archive)
files.download(str(archive))


## Interpretation

- Ordinary parallel/sequential and identity checks establish the baseline numerical comparison.
- Empty quantized-cache prefill is tested separately from later decode reads.
- A one-token answer exercises prefill only, so it should have no decode dequantization reads.
- The trace checks reconstructed prefixes and that the SDPA interface receives the exact cache-returned K/V objects.
- Generation replay checks the emitted greedy tokens against teacher forcing under the same cache policy.
- Identical quantized and ordinary scores are permitted. Positive recovery is not a gate.

Still outside this run: padded/multi-example batches, FP16 validation, full TOFU integration, statistical confirmation, and final scientific claims. B's residual-boundary runs remain separate supporting evidence.

Local checks used Transformers 5.5.4 with a tiny random CPU Llama and a simple rounding cache. They do not validate HQQ or the T4; this returned run will provide that evidence.